[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/04_human_evaluation/04_human_evaluation.ipynb)

# 04 · 人类评测方法论（从零实现）

目标：把**人评**当成一台需要校准的测量仪器。用 numpy 从零：① Likert 聚合 + 评分者去偏；② 成对胜负→胜率矩阵；③ **Bradley-Terry MLE**（核心，梯度上升拟合实力分、恢复真实排名）；④ 顺序/位置偏置的模拟与校正；⑤ 长度偏置——回归扣除；⑥ bootstrap 给实力分/胜率加 CI。

路线：Likert 去偏 → 胜率矩阵 → **Bradley-Terry** → 顺序偏置 → 长度偏置 → bootstrap CI → ✏️ 练习 → 📖 答案 → 🧪 真实 Arena 偏好胶囊。

> 纪律：所有随机用 `default_rng(seed)`；所有「应成立的性质」写 `assert`（去偏后排名更准、BT 恢复真实排名、校正后偏置趋近 0、CI 覆盖真值……）。

## 0 · 数据 helper（联网取真实数据，失败回退）

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · Likert 聚合与评分者去偏

评分者有不同的内心标尺（**尺度差异**）：甲整体打高、乙整体打低，会污染对象均分。

**去偏**：减去每个评分者自己的均值，只保留他对对象之间的*相对*判断。我们造一个「评分者有系统偏置但排序一致」的数据，验证去偏后的对象排名比朴素均分**更接近真实质量排名**。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 真实质量(5个对象)；评分者排序一致，但各有 加性偏置 + 小噪声
n_items, n_raters = 5, 8
true_quality = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
rater_bias = rng.normal(0, 1.2, n_raters)           # 每个评分者的整体宽严
# scores[r, i] = 真实质量 + 评分者偏置 + 噪声
scores = (true_quality[None, :] + rater_bias[:, None]
          + rng.normal(0, 0.3, (n_raters, n_items)))

def naive_item_means(scores):
    return scores.mean(axis=0)                       # 直接对评分者平均

def debiased_item_means(scores):
    centered = scores - scores.mean(axis=1, keepdims=True)  # 减每个评分者自己的均值
    return centered.mean(axis=0)

def rank_of(x):
    # 返回名次(从小到大)；用 argsort 的 argsort
    return np.argsort(np.argsort(x))

naive = naive_item_means(scores)
deb   = debiased_item_means(scores)
true_rank = rank_of(true_quality)
err_naive = np.abs(rank_of(naive) - true_rank).sum()
err_deb   = np.abs(rank_of(deb) - true_rank).sum()
print('真实质量排名 :', true_rank)
print('朴素均分排名 :', rank_of(naive), ' 排名误差=', err_naive)
print('去偏后排名   :', rank_of(deb),   ' 排名误差=', err_deb)
# 去偏移除了评分者加性偏置，对象间相对序应当恢复
assert np.array_equal(rank_of(deb), true_rank), '去偏后应恢复真实排名'
print('✅ 去中心化扣掉了评分者宽严，对象排名回到真实序')

## 2 · 成对胜负 → 胜率矩阵

成对比较产生一堆 `(i, j, winner)` 记录。先把它们聚合成两个矩阵：`wins[i,j]` = i 战胜 j 的次数，`games[i,j]` = i 与 j 的总对局数。

胜率矩阵 `winrate[i,j] = wins[i,j]/games[i,j]`。这是 Bradley-Terry 的输入。

In [ ]:
def tally(battles, n):
    '''battles: list of (i, j, y)；y=1 表示 i 胜, y=0 表示 j 胜。返回 wins, games 矩阵。'''
    wins = np.zeros((n, n)); games = np.zeros((n, n))
    for i, j, y in battles:
        games[i, j] += 1; games[j, i] += 1
        if y == 1:
            wins[i, j] += 1
        else:
            wins[j, i] += 1
    return wins, games

# 造一批对局: 4 个对象, 真实强弱 0<1<2<3
n = 4
battles = [(0,1,0),(0,1,0),(1,2,0),(2,3,0),(0,3,0),(1,3,0),
           (0,2,0),(2,3,0),(1,2,1),(0,1,1)]   # 多数符合 0<1<2<3, 个别翻盘
wins, games = tally(battles, n)
with np.errstate(invalid='ignore', divide='ignore'):
    winrate = np.where(games > 0, wins / games, np.nan)
print('wins 矩阵:'); print(wins.astype(int))
print('games 矩阵:'); print(games.astype(int))
# 对称性: games[i,j]==games[j,i]; wins[i,j]+wins[j,i]==games[i,j]
assert np.array_equal(games, games.T)
assert np.allclose(np.nan_to_num(wins + wins.T), games)
# 对象3(最强)对其它的总胜率应高于对象0(最弱)
wr3 = np.nansum(wins[3]) / np.nansum(games[3])
wr0 = np.nansum(wins[0]) / np.nansum(games[0])
print(f'对象3 总胜率={wr3:.2f} > 对象0 总胜率={wr0:.2f}')
assert wr3 > wr0
print('✅ 胜负聚合正确：wins+winsᵀ=games，强者总胜率更高')

## 3 · ⭐ Bradley-Terry：MLE 拟合实力分（核心）

模型：`P(i 胜 j) = σ(s_i − s_j)`。给定胜负，用**梯度上升**最大化对数似然拟合实力分 `s`。

梯度有干净形式：`∂L/∂s_i = (i 实际胜场) − Σ_j games[i,j]·σ(s_i − s_j)` = 观测胜场 − 期望胜场。
每步更新后**中心化 `s` 到均值 0**（固定标度，因为只有差可辨识）。

验证：从**已知真实 `s`** 造数据，拟合后**恢复出的排名与真实排名完全一致**，且 `s` 与真值高度相关。

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def bradley_terry_mle(wins, games, lr=0.1, n_iter=2000):
    '''梯度上升拟合 Bradley-Terry 实力分 s（中心化到均值0）。'''
    n = wins.shape[0]
    s = np.zeros(n)
    win_counts = wins.sum(axis=1)                  # 每个对象的实际总胜场
    for _ in range(n_iter):
        # 期望胜场: 对每个 j, games[i,j]*sigmoid(s_i - s_j)
        diff = s[:, None] - s[None, :]             # diff[i,j] = s_i - s_j
        P = sigmoid(diff)                          # 预测 i 胜 j 的概率
        expected = (games * P).sum(axis=1)         # 期望总胜场
        grad = win_counts - expected               # 观测 - 期望
        s += lr * grad / max(games.sum(), 1)       # 归一化步长, 稳定
        s -= s.mean()                              # 固定标度
    return s

# 造数据: 已知真实实力分, 按 BT 概率生成大量对局
rng = np.random.default_rng(1)
n = 6
true_s = np.array([-2.0, -1.0, 0.0, 0.5, 1.5, 2.5])
true_s = true_s - true_s.mean()
G = 400                                            # 每对对局数
wins = np.zeros((n, n)); games = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        p = sigmoid(true_s[i] - true_s[j])
        wi = rng.binomial(G, p)                     # i 胜的场数
        wins[i, j] += wi; wins[j, i] += G - wi
        games[i, j] += G; games[j, i] += G

s_hat = bradley_terry_mle(wins, games)
print('真实实力分 :', np.round(true_s, 3))
print('拟合实力分 :', np.round(s_hat, 3))
# 1) 排名完全一致
assert np.array_equal(np.argsort(s_hat), np.argsort(true_s)), 'BT 应恢复真实排名'
# 2) s 与真值高度相关(中心化后)
corr = np.corrcoef(s_hat, true_s)[0, 1]
print(f'拟合 s 与真实 s 的相关 = {corr:.4f}')
assert corr > 0.99, 's 应与真值高度相关'
print('✅ Bradley-Terry MLE 从胜负恢复了真实实力分与排名')

## 4 · 顺序/位置偏置：模拟与校正

评委有**位置偏置**：倾向于选「第一个」选项，与质量无关。若总把同一模型放左边，它会白捡红利。

模型：`P(选左) = clip(真实左胜率 + δ)`，δ 是位置效应。**对策**：左右随机化，正反各评——位置效应在平均时抵消。验证：朴素(固定位置)估计被 δ 带偏；随机化后估计 ≈ 真实胜率。

In [ ]:
rng = np.random.default_rng(2)
true_left_winrate = 0.55      # A(左) 对 B 的真实胜率
delta = 0.12                  # 位置偏置: 无脑偏向左边 +0.12
N = 8000

# 情形A: A 永远在左 -> 观测胜率被 +delta 带偏
p_fixed = np.clip(true_left_winrate + delta, 0, 1)
obs_fixed = (rng.random(N) < p_fixed).mean()        # A 的观测胜率(固定在左)

# 情形B: 左右随机化, 正反各半。记录'A 是否胜', 校正位置效应
A_on_left = rng.random(N) < 0.5
# 当 A 在左: P(A胜)=clip(true+delta); 当 A 在右: P(A胜)=clip(true-delta)
pA = np.where(A_on_left, np.clip(true_left_winrate+delta,0,1),
                          np.clip(true_left_winrate-delta,0,1))
A_wins = rng.random(N) < pA
obs_random = A_wins.mean()                          # 随机化下 A 的总胜率
# 校正: 估计 delta = (在左胜率 - 在右胜率)/2, 再从'在左'里扣掉
wr_left  = A_wins[A_on_left].mean()
wr_right = A_wins[~A_on_left].mean()
delta_hat = (wr_left - wr_right) / 2
corrected = wr_left - delta_hat                     # 扣掉位置效应
print(f'真实胜率           = {true_left_winrate:.3f}')
print(f'朴素(A固定在左)估计 = {obs_fixed:.3f}  (被位置偏置 +{delta} 带偏)')
print(f'随机化总胜率        = {obs_random:.3f}  (位置效应已大致抵消)')
print(f'估计的位置偏置 δ̂   = {delta_hat:.3f}  (真值 {delta})')
print(f'校正后估计          = {corrected:.3f}')
assert abs(obs_fixed - true_left_winrate) > 0.05, '固定位置应明显有偏'
assert abs(obs_random - true_left_winrate) < 0.02, '随机化应抵消位置效应'
assert abs(delta_hat - delta) < 0.03, '应估出位置偏置大小'
print('✅ 左右随机化 + 校正：把被位置偏置污染的胜率拉回真实值')

## 5 · 长度偏置：回归扣除

评委(尤其 LLM judge)偏好**更长**的回答，与质量无关。当「质量本应相同、只是更长」时，长的一方仍系统性胜出。

我们构造质量相同、长度不同的成对样本，胜负 = `σ(β·Δlen)`（纯长度偏置）。用**回归**估 β，把长度能解释的部分扣掉，验证残余的「质量效应」趋近 0（真值）。

In [ ]:
rng = np.random.default_rng(3)
N = 4000
# 每对: A、B 质量相同(真实质量差=0); 长度差 dlen 随机
dlen = rng.normal(0, 1.0, N)            # A 相对 B 的(标准化)长度差
beta_true = 0.8                         # 评委每多1单位长度, logit +0.8
true_quality_diff = 0.0                 # 质量真的没差别
logit = true_quality_diff + beta_true * dlen
A_wins = (rng.random(N) < sigmoid(logit)).astype(float)

naive_winrate = A_wins.mean()           # 朴素: A 的总胜率
# 朴素胜率与 dlen 的均值有关; 若 dlen 均值>0 会偏离 0.5
print(f'长度差均值 = {dlen.mean():.3f}, A 朴素胜率 = {naive_winrate:.3f}')

# 用 logistic 回归(梯度上升)估 [intercept(质量效应), beta(长度效应)]
def logreg_fit(X, y, lr=0.3, n_iter=3000):
    w = np.zeros(X.shape[1])
    for _ in range(n_iter):
        p = sigmoid(X @ w)
        grad = X.T @ (y - p) / len(y)
        w += lr * grad
    return w

X = np.column_stack([np.ones(N), dlen])  # [截距, 长度差]
w = logreg_fit(X, A_wins)
intercept, beta_hat = w
print(f'回归: 质量效应(截距)={intercept:.3f} (真值≈0), 长度效应 β̂={beta_hat:.3f} (真值 {beta_true})')
# 截距(扣掉长度后的质量效应)应接近 0; beta_hat 应接近真值
assert abs(beta_hat - beta_true) < 0.15, '应估出长度偏置 β'
assert abs(intercept) < 0.15, '扣掉长度后, 质量效应应≈0(本就没质量差)'
print('✅ 回归把长度能解释的胜负扣掉，露出真实(为零)的质量效应')

## 6 · bootstrap 给实力分/胜率加置信区间

排名上第 3 和第 4 名，如果误差棒糊在一起，名次差异就不可信。

对**对局**有放回重采样，每次**重拟合 Bradley-Terry**，得到实力分的 bootstrap 分布 → CI。验证 CI 覆盖真值，且实力相近的两个对象 CI 重叠、实力悬殊的不重叠。

In [ ]:
rng = np.random.default_rng(4)
# 造对局列表(展开成单场), 真实 s
n = 4
true_s = np.array([-1.5, -0.2, 0.2, 1.5]); true_s = true_s - true_s.mean()
battles = []
G = 150
for i in range(n):
    for j in range(i+1, n):
        p = sigmoid(true_s[i] - true_s[j])
        for _ in range(G):
            y = 1 if rng.random() < p else 0
            battles.append((i, j, y))
battles = np.array(battles)

def fit_s_from_battles(bat, n):
    wins, games = tally(bat, n)
    return bradley_terry_mle(wins, games)

def bootstrap_bt(battles, n, n_boot=300, seed=0):
    r = np.random.default_rng(seed)
    M = len(battles)
    S = np.zeros((n_boot, n))
    for b in range(n_boot):
        idx = r.integers(0, M, M)
        S[b] = fit_s_from_battles(battles[idx], n)
    return S

S = bootstrap_bt(battles, n, n_boot=300, seed=7)
lo, hi = np.percentile(S, [2.5, 97.5], axis=0)
point = fit_s_from_battles(battles, n)
for i in range(n):
    print(f'对象{i}: s={point[i]:+.2f}  95%CI=[{lo[i]:+.2f},{hi[i]:+.2f}]  真值={true_s[i]:+.2f}')
# CI 应覆盖真值(大多数)
covered = ((lo <= true_s) & (true_s <= hi)).sum()
assert covered >= n - 1, 'CI 应覆盖绝大多数真值'
# 最强(3)与最弱(0) CI 不重叠; 但中间相近的(1,2)可能重叠
assert lo[3] > hi[0], '实力悬殊的对象 CI 不应重叠'
print('✅ bootstrap CI 覆盖真值；实力悬殊者可区分，相近者(看CI是否重叠)需谨慎')

---
## ✏️ 练习 1：Likert 评分者去偏 + z 标准化

更彻底的去偏不仅减均值，还**除以各评分者的标准差**（z 标准化），消除「有人用满量程、有人只用中间」的尺度差异。

实现 `zscore_debias(scores)`：对每个评分者(行)做 `(x - mean)/std`，再对评分者平均，返回每个对象的去偏分数。

In [ ]:
def zscore_debias(scores):
    # TODO: 对每一行(评分者) 做 (x - 行均值)/行标准差, 再 axis=0 平均
    #       注意 std 用 ddof=0; 防止除0(本练习数据 std>0)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng = np.random.default_rng(10)
true_q = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
# 评分者: 不同偏置 + 不同尺度(乘性)
bias = rng.normal(0, 1, 6); scale = rng.uniform(0.5, 2.0, 6)
scores = scale[:,None]*(true_q[None,:]) + bias[:,None] + rng.normal(0,0.2,(6,5))
out = zscore_debias(scores)
assert out.shape == (5,)
assert np.array_equal(np.argsort(out), np.argsort(true_q)), 'z 去偏后应恢复真实排名'
print('z 去偏分数:', np.round(out,3))
print('✅ 练习 1 通过：z 标准化同时消除评分者的偏置与尺度差异')

## ✏️ 练习 2：胜率矩阵 → Bradley-Terry

给定 `wins, games` 矩阵，实现 `fit_bt(wins, games)` 返回中心化的实力分（复用第 3 节的梯度上升思路）。

要求能从胜负恢复真实排名。

In [ ]:
def fit_bt(wins, games, lr=0.1, n_iter=2000):
    # TODO: 梯度上升拟合 Bradley-Terry 实力分
    #   s 初始化为 0; win_counts = wins.sum(axis=1)
    #   每步: diff[i,j]=s_i-s_j; P=sigmoid(diff); expected=(games*P).sum(1)
    #         grad=win_counts-expected; s += lr*grad/games.sum(); s -= s.mean()
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng = np.random.default_rng(11)
n = 5
ts = np.array([-2.,-1.,0.,1.,2.]); ts -= ts.mean()
wins = np.zeros((n,n)); games = np.zeros((n,n)); G=300
for i in range(n):
    for j in range(i+1,n):
        p = sigmoid(ts[i]-ts[j]); wi = rng.binomial(G,p)
        wins[i,j]+=wi; wins[j,i]+=G-wi; games[i,j]+=G; games[j,i]+=G
s = fit_bt(wins, games)
assert np.array_equal(np.argsort(s), np.argsort(ts)), 'BT 应恢复真实排名'
assert np.corrcoef(s, ts)[0,1] > 0.99
print('拟合实力分:', np.round(s,3)); print('✅ 练习 2 通过：胜率矩阵→BT 实力分')

## ✏️ 练习 3：长度偏置校正

给定每对的长度差 `dlen` 与胜负 `y`（A 是否胜），以及一个**真实质量差** `q`，胜负 = `σ(q + β·dlen)`。

实现 `recover_quality_effect(dlen, y)`：用 logistic 回归 `[1, dlen]` 拟合，返回 `(质量效应=截距, 长度效应=β)`。

In [ ]:
def recover_quality_effect(dlen, y, lr=0.3, n_iter=3000):
    # TODO: X=[ones, dlen]; 梯度上升拟合 logistic 回归; 返回 (截距, beta)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng = np.random.default_rng(12)
N = 5000
dlen = rng.normal(0, 1, N)
q_true, beta_true = 0.5, 1.0       # 这次 A 真的更好(q=0.5) 且更长
y = (rng.random(N) < sigmoid(q_true + beta_true*dlen)).astype(float)
q_hat, beta_hat = recover_quality_effect(dlen, y)
print(f'质量效应={q_hat:.3f} (真值 {q_true}), 长度效应={beta_hat:.3f} (真值 {beta_true})')
assert abs(q_hat - q_true) < 0.15, '应恢复真实质量效应'
assert abs(beta_hat - beta_true) < 0.2, '应恢复长度效应'
# 朴素胜率会把长度红利算进质量; 回归把两者分开
print('✅ 练习 3 通过：把质量效应与长度偏置分离')

## ✏️ 练习 4：胜率的 bootstrap CI

给定 A 对 B 的一串胜负 `outcomes`(0/1)，实现 `winrate_ci(outcomes, n_boot, seed)`：
bootstrap 重采样算胜率，返回 `(点估计, lo, hi)`（95%）。再判断该胜率是否**显著不同于 0.5**（CI 不含 0.5）。

In [ ]:
def winrate_ci(outcomes, n_boot=2000, seed=0):
    # TODO: 对 outcomes 有放回重采样 n_boot 次, 每次算 mean;
    #   返回 (点估计=outcomes.mean(), lo=2.5百分位, hi=97.5百分位)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng = np.random.default_rng(13)
# 真胜率 0.62, 1000 场 -> 应显著 > 0.5
out = (rng.random(1000) < 0.62).astype(int)
pt, lo, hi = winrate_ci(out, n_boot=2000, seed=1)
print(f'胜率={pt:.3f}, 95%CI=[{lo:.3f},{hi:.3f}]')
assert lo <= pt <= hi and 0.0 <= lo < hi <= 1.0
assert not (lo <= 0.5 <= hi), '0.62、1000场 应显著区别于0.5(CI不含0.5)'
# 真胜率 0.5 的小样本 -> CI 应含 0.5(区分不开)
out2 = (rng.random(80) < 0.5).astype(int)
pt2, lo2, hi2 = winrate_ci(out2, n_boot=2000, seed=2)
assert lo2 <= 0.5 <= hi2, '0.5、小样本 应区分不开(CI 含0.5)'
print('✅ 练习 4 通过：胜率 CI + 是否显著区别于随机(0.5)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def zscore_debias(scores):
    mu = scores.mean(axis=1, keepdims=True)
    sd = scores.std(axis=1, keepdims=True, ddof=0)
    z = (scores - mu) / sd
    return z.mean(axis=0)

In [ ]:
# 练习 2 参考答案
def fit_bt(wins, games, lr=0.1, n_iter=2000):
    n = wins.shape[0]; s = np.zeros(n)
    win_counts = wins.sum(axis=1)
    for _ in range(n_iter):
        diff = s[:, None] - s[None, :]
        P = sigmoid(diff)
        expected = (games * P).sum(axis=1)
        grad = win_counts - expected
        s += lr * grad / max(games.sum(), 1)
        s -= s.mean()
    return s

In [ ]:
# 练习 3 参考答案
def recover_quality_effect(dlen, y, lr=0.3, n_iter=3000):
    N = len(y); X = np.column_stack([np.ones(N), dlen]); w = np.zeros(2)
    for _ in range(n_iter):
        p = sigmoid(X @ w)
        grad = X.T @ (y - p) / N
        w += lr * grad
    return w[0], w[1]

In [ ]:
# 练习 4 参考答案
def winrate_ci(outcomes, n_boot=2000, seed=0):
    r = np.random.default_rng(seed)
    outcomes = np.asarray(outcomes); M = len(outcomes)
    boots = np.array([outcomes[r.integers(0, M, M)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(outcomes.mean()), float(lo), float(hi)

## 🧪 真实数据胶囊：真实 Chatbot Arena 风格偏好

用真实的成对人类偏好数据给模型排名。尝试取 LMSYS **MT-Bench 人类裁判**数据(`lmsys/mt_bench_human_judgments`，公开非 gated；字段含 `model_a`、`model_b`、`winner`)，聚合成胜负、拟合 Bradley-Terry。

**联网取真实对战；失败回退到内置的真实对战记录**(基于公开 Arena 大致强弱：强模型 > 中 > 弱)。

In [ ]:
def load_arena_battles(n=600):
    '''取真实 Arena 成对偏好; 失败回退内置真实对战。返回 (battles_df, models, source)。
       battles_df: 列 [model_a, model_b, winner(model_a/model_b/tie)]。'''
    try:
        # LMSYS MT-Bench 人类裁判: 真实成对偏好, 公开非 gated; 字段 model_a/model_b/winner
        rows = hf_rows('lmsys/mt_bench_human_judgments', 'default', 'human', n)
        recs = []
        for r in rows:
            wa, wb, w = r.get('model_a'), r.get('model_b'), r.get('winner')
            if wa and wb and w in ('model_a','model_b','tie') and wa != wb:
                recs.append((wa, wb, w))
        if len(recs) >= 50:
            df = pd.DataFrame(recs, columns=['model_a','model_b','winner'])
            models = sorted(set(df.model_a) | set(df.model_b))
            return df, models, 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 回退: 内置真实对战(4个模型, 真实大致强弱 strong>mid>weak>tiny)
    r = np.random.default_rng(0)
    models = ['gpt-4', 'claude-1', 'vicuna-13b', 'alpaca-13b']  # 真实 Arena 模型名
    strength = {'gpt-4':2.0, 'claude-1':1.0, 'vicuna-13b':-0.5, 'alpaca-13b':-2.0}
    recs = []
    for _ in range(600):
        a, b = r.choice(models, 2, replace=False)
        pa = 1/(1+np.exp(-(strength[a]-strength[b])))
        recs.append((a, b, 'model_a' if r.random() < pa else 'model_b'))
    df = pd.DataFrame(recs, columns=['model_a','model_b','winner'])
    return df, models, 'builtin'

df, models, source = load_arena_battles(600)
print(f'数据来源={source}; {len(df)} 场对战, {len(models)} 个模型')
print(df.head(3).to_string(index=False))
assert len(df) >= 50 and len(models) >= 2
print('✅ 拿到真实(或回退)成对偏好对战')

**用 Bradley-Terry 给真实模型排名**：把 winner 转成 0/1，聚合 wins/games，拟合实力分，输出排行榜。

In [ ]:
idx = {m:i for i,m in enumerate(models)}
battles = []
for _, row in df.iterrows():
    i, j = idx[row.model_a], idx[row.model_b]
    if row.winner == 'model_a':
        battles.append((i, j, 1))
    elif row.winner == 'model_b':
        battles.append((i, j, 0))
    # tie 跳过(简单起见)
battles = np.array(battles)
wins, games = tally(battles, len(models))
s = bradley_terry_mle(wins, games)
order = np.argsort(-s)
print('Bradley-Terry 排行榜:')
for rank, k in enumerate(order, 1):
    print(f'  #{rank}  {models[k]:14s}  实力分={s[k]:+.3f}')
assert len(s) == len(models) and abs(s.mean()) < 1e-6  # 中心化
print('✅ 从真实成对偏好拟合出模型排行榜')

**🧪 胶囊练习**：实现 `model_winrate(df, models, a, b)`——直接从对战记录算模型 a 对模型 b 的**实际胜率**（忽略 tie），用它对拍 Bradley-Terry 预测的胜率 `σ(s_a − s_b)` 大方向一致。

In [ ]:
def model_winrate(df, models, a, b):
    # TODO: 统计 a 与 b 直接对局中 a 胜的比例(winner==model_a 当 a 在 model_a 列, 反之亦然)
    #   遍历 df, 只看 {a,b} 的对局, 返回 a 胜场 / (a胜+b胜)
    raise NotImplementedError

In [ ]:
# 自测
# 取实力分差最大的一对, 实际胜率应明显>0.5 且与 BT 预测同向
top, bottom = models[order[0]], models[order[-1]]
wr = model_winrate(df, models, top, bottom)
bt_pred = 1/(1+np.exp(-(s[idx[top]] - s[idx[bottom]])))
print(f'{top} vs {bottom}: 实际胜率={wr:.2f}, BT预测={bt_pred:.2f}')
assert wr > 0.5 and bt_pred > 0.5, '最强对最弱, 实际与预测都应>0.5'
print('✅ 胶囊练习通过：实际胜率与 Bradley-Terry 预测同向')

In [ ]:
# 📖 胶囊参考答案
def model_winrate(df, models, a, b):
    a_wins = b_wins = 0
    for _, row in df.iterrows():
        pair = {row.model_a, row.model_b}
        if pair != {a, b}:
            continue
        winner_model = row.model_a if row.winner == 'model_a' else (row.model_b if row.winner == 'model_b' else None)
        if winner_model == a: a_wins += 1
        elif winner_model == b: b_wins += 1
    return a_wins / (a_wins + b_wins) if (a_wins + b_wins) > 0 else 0.5

### 小结
- **人评不是金标准，它本身是一次测量**：要查偏置、量信度、报 CI。
- **Likert** 受评分者尺度差异污染 → 去中心化/z 标准化；有序≠等距，慎求平均。
- **成对比较**抗尺度偏置、方差低（Arena 用它）；**Bradley-Terry** 用 MLE 把胜负拼成实力分排名。
- **顺序偏置**靠左右随机化抵消；**长度偏置**靠回归扣除——人和 LLM judge 都中招。
- **信度(ICC)** 看真实信号占比；**bootstrap CI** 给实力分/胜率加误差棒，名次差异要穿透 CI。

下一站：**模块 05 · 心理测量与 IRT** —— 把「题目有多难」和「模型有多强」分开建模，比 Bradley-Terry 更进一步。